In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
schema = ['uid','device','song_id','song_type','song_name','singer','play_time','song_length','paid_flag','file_name','label', 'date']

In [ ]:
df_play = pd.read_csv('/Users/ZhijingYe/Desktop/data/output/play_sample_log.csv',
                 delimiter='\t',index_col=False,
                 dtype = {'uid':'str', 'song_id':'str','song_type' : 'str'})


In [ ]:
df_play.head()

In [ ]:
df_play = df_play.drop(df_play.columns[0], axis=1)
df_play.head()

In [ ]:
df_play = df_play.drop(df_play.columns[0], axis=1)
df_play.head()

In [ ]:
df_play.isnull().sum(axis = 0)

### 1. file_name and paid_flag
First drop file_name and paid_flag columns. 

(a) there is a column 'date' contains log date information <br>
(b) In data cleaning file, we noticed there is no real paid users so this columns can be dropped

In [ ]:
df_play = df_play.drop(['paid_flag', 'file_name'], axis=1)

In [ ]:
df_play.head()

In [ ]:
df_play.info()

### 2. uid

In [ ]:
df_play.uid.value_counts()

#### There are some users have way more than usual number of logs.
#### Say one log takes one min, there are 24*60 = 1440 mins per day. Thoes uses have more than 1440 logs per date should be robots.


In [ ]:
res = df_play[['uid','date','song_id']].groupby(['uid','date']).count().rename(columns={'song_id': 'count'})

In [ ]:
res.reset_index()

In [ ]:
res2 = res.reset_index()

In [ ]:
res22 = res2.loc[res2.uid>0]

In [ ]:
res22.head(10)

In [ ]:
#res1 = df_play.loc[df_play.uid > 0]

In [ ]:
#res1.head(10)

In [ ]:
#res.head(20)

In [ ]:
#res1.tail(10)

In [ ]:
res_agg = res22.groupby('uid')['count'].agg(['sum','count'])

In [ ]:
res_agg.head()

In [ ]:
res_agg['avg_logs'] = res_agg['sum']/res_agg['count']

In [ ]:
res_agg.head()

In [ ]:
uid_robot = res_agg[res_agg['avg_logs'] > 1440].index

In [ ]:
uid_robot

In [ ]:
uid_robot = set(uid_robot)

In [ ]:
uid_robot

In [ ]:
# Remove those suspecious robot users
df_play = df_play[~df_play['uid'].isin(uid_robot)]

In [ ]:
df_play.shape

In [ ]:
del res22, res_agg

### 3. Device type was already taken care in data cleansing step.

In [ ]:
df_play.device.value_counts()

### 4. Date

In [ ]:
df_play.groupby('date').size()

#### Convert date to string and remove the three outlier: 0, 168, 169, 238, 260

In [ ]:
filter_date = set([0, 168, 169, 238, 260])

In [ ]:
df_play = df_play[~df_play['date'].isin(filter_date)]

In [ ]:
df_play['date'] = df_play['date'].astype('string')

In [ ]:
df_play[['date']].info()

In [ ]:
# Check the churn rate for different date
df_play.groupby(['date','label']).size()

#### 5. Song type

In [ ]:
df_play.song_type.unique()

In [ ]:
df_play.song_type = df_play.song_type.apply(lambda x: str(x).split('(')[0].rstrip())

In [ ]:
df_play.song_type.value_counts()

In [ ]:
# There are some missing values in song_type, impute with most common song_type 0
df_play.loc[df_play['song_type'] == '',['song_type']] = '0'

In [ ]:
df_play.song_type.value_counts()

#### 6. Label

In [ ]:
df_play.label.value_counts()

In [ ]:
# Convert label to int
df_play['label'] = df_play['label'].apply(lambda x: str(x).split('.')[0])

In [ ]:
df_play.label.value_counts()

#### 7. Play time

play_time are strings. Some are has '>' in it. Split with '>' and keep only the part in front of '>'. 

Then, change the type to float

In [ ]:
df_play.play_time.value_counts()

In [ ]:
df_play['play_time'] = df_play['play_time'].astype('string').apply(lambda x: x.split('>')[0])
df_play.play_time.value_counts()

In [ ]:
df_play['play_time'] = df_play['play_time'].convert_objects(convert_numeric= True)

In [ ]:
df_play[['play_time']].info()

#### 8. Song_id
song_id should be an integer or a string of numbers without decimal

The corner case is where song_id == '0'. There are more than 500k logs with '0' as song_id, but they 
are actually different songs:

    Some songs have '0' and another song_id

    Some songs only have '0' as song_id

song_id imputation:

    1) change the song_id to its most frequent appeared values, if there is value other than '0'

    2) if it doesn't have another song_id, create new song_ids.

What is needed is a {(song_name, singer, song_type):song_id} dictionary.

In [ ]:
df_play.song_id = df_play.song_id.apply(lambda x: x.split('.')[0].rstrip())

In [ ]:
df_play.loc[df_play.song_id == '0']

In [ ]:
df_play.groupby(['song_name','singer','song_type']).size().shape

In [ ]:
df_play.groupby(['song_name','singer','song_type','song_id']).size().shape

Although the combination of ('song_name','singer','song_type') can identify a song without ambiguity, there 
ISN'T an one-to-one mapping from ('song_name','singer','song_type') to 'song_id'.

Find out the duplicated 'song_id's

In [ ]:
df_play[['song_name']].info()

In [ ]:
##df['song_name'] = df['song_name'].astype('str') 

In [ ]:
# Notice there are some empty spaces as inputs in song_name and singer, strip tailoring spaces
df_play[['song_name', 'singer']] = df_play[['song_name', 'singer']].applymap(lambda x: str(x).rstrip() if str(x) else x)

In [ ]:
df_play.isnull().sum(axis = 0)

In [ ]:
# Then remove the logs with nan song_name and singer, totally 58 logs
df_play = df_play.loc[df_play.song_name.notnull() & \
                      df_play.singer.notnull()]
df_play.isnull().sum(axis = 0)

In [ ]:
# Count number of logs groupped by song_name, singer and song_type 
song_id_groupby_df = df_play[['song_name','singer','song_type','song_id']]\
                        .groupby(['song_name','singer','song_type']).count().rename(columns={'song_id': 'count'})

In [ ]:
# make new song_id with the play times rank, where more played song is ranked as 1st.
song_id_groupby_df['song_id_new'] = song_id_groupby_df['count'].rank(method = 'first', ascending = False).astype('int')

In [ ]:
song_id_groupby_df.reset_index(inplace = True)

In [ ]:
song_id_groupby_df.info()

In [ ]:
song_id_groupby_df.head(10)

In [ ]:
song_id_groupby_df.loc[2, 'song_name']

In [ ]:
# Merge new_song_id to create a new dataframe
df_play_log = pd.merge(df_play, song_id_groupby_df,  how='left', \
                       left_on=['song_name','singer','song_type'], \
                       right_on = ['song_name','singer','song_type'])

In [ ]:
df_play_log.head()

In [ ]:
df_play_log.shape, df_play.shape

In [ ]:
df_play_log['song_id_new'].isnull().sum()

In [ ]:
# get favorite songs info
popular_songs = df_play_log['song_id_new'].value_counts()[:200].index

popular_songs[100:110]

In [ ]:
# the rarely played songs are also worth examining.
rarely_played = sum(df_play_log['song_id'].value_counts()<2)
rarely_played

In [ ]:
rarely_played_songs = df_play_log.song_id_new.value_counts()[-1:-rarely_played-1:-1].index
len(rarely_played_songs)

In [ ]:
# searching play log of a certain rare song
df_play_log[df_play_log.song_id_new == rarely_played_songs[0]]

In [ ]:
song_id_groupby_df[song_id_groupby_df.song_id_new == rarely_played_songs[0]]

### 9.singer

In [ ]:
singer_groupby_df = df_play[['singer','song_id']]\
                   .groupby(['singer']).count().rename(columns={'song_id': 'count'})

In [ ]:
singer_groupby_df.reset_index(inplace = True)
singer_groupby_df.head()

In [ ]:
singer_groupby_df.info()

In [ ]:
singer_groupby_df['singer_popularity_rank'] = singer_groupby_df[['count']].rank(method = 'first', ascending = False).astype(int)

In [ ]:
singer_groupby_df.head()

In [ ]:
# Merge singer_popularity to create a new dataframe
df_play_log = pd.merge(df_play_log, singer_groupby_df,  how='left', \
                       left_on=['singer'], \
                       right_on = ['singer'])

In [ ]:
df_play_log.head()

In [ ]:
df_play_log.info()

In [ ]:
# get favorite songs info
popular_singers = df_play_log['singer_popularity_rank'].value_counts()[:200].index

popular_singers[100:110]

### reduce the size of the df_play_log
In the following analysis, the original song_id, song_names and singer can be ommitted from the df_play_log, while the only information left is song_id_new, singer_popularity_rank.

In [ ]:
df_play_log = df_play_log.drop(['song_name','singer','count_x','count_y'],axis = 1)
df_play_log.head()

In [ ]:
df_play_log.drop(['song_id'], axis = 1, inplace = True)
df_play_log.head()

#### Save current data frames


In [ ]:
play_file = '/Users/ZhijingYe/Desktop/data/output/reduced_play_log.csv'
df_play_log.to_csv(play_file,sep = '\t', mode = 'a', encoding = 'utf-8')

In [ ]:
# also save into .pickle
play_file_pickle = '/Users/ZhijingYe/Desktop/data/output/reduced_play_log.pkl'
df_play_log.to_pickle(play_file_pickle)

In [ ]:
# songs
song_file = '/Users/ZhijingYe/Desktop/data/output/songs.csv'
song_id_groupby_df.to_csv(song_file,sep = '\t', mode = 'a')

In [ ]:
song_file_pickle = '/Users/ZhijingYe/Desktop/data/output/songs.pkl'
song_id_groupby_df.to_csv(song_file_pickle)

In [ ]:
singer_file = '/Users/ZhijingYe/Desktop/data/output/singers.csv'
singer_groupby_df.to_csv(singer_file,sep = '\t', mode = 'a')

In [ ]:
singer_file_pickle = '/Users/ZhijingYe/Desktop/data/output/singers.pkl'
singer_groupby_df.to_csv(singer_file_pickle)